# Table Models for Observability Data

For observability workloads, the DUPLICATE model is recommended for raw logs, traces, and metrics. This model preserves all detail data, provides the fastest write performance, and is the default table model when no model is specified.

The aggregate and unique models are also widely used in other observability scenarios.

## 1. Aggregate model for ERROR logs

Imagine a scenario where, in addition to a large volume of INFO logs, many ERROR logs are generated every day. These ERROR logs need to be analyzed by day and by `service_name` to identify the specific errors reported for each service on each day.

You could query the original logs table directly, filter for ERROR entries, and then group the results by day and `service_name`. However, this approach can be slow.

To solve this problem, you can filter the ERROR records when writing the original logs data to a dedicated ERROR table and use the aggregate model:

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [1]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [2]:
lab.execute(r"""
CREATE TABLE `otel_logs_for_error` (
  `timestamp` datetime(6) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200) REPLACE NULL,
  `trace_id` varchar(200) REPLACE NULL,
  `span_id` text REPLACE NULL,
  `severity_number` int REPLACE NULL,
  `severity_text` text REPLACE NULL,
  `body` text REPLACE NULL,
  `resource_attributes` variant REPLACE NULL,
  `log_attributes` variant REPLACE NULL,
  `scope_name` text REPLACE NULL,
  `scope_version` text REPLACE NULL,
  `count_value` bigint SUM NULL
) ENGINE=OLAP
AGGREGATE KEY(`timestamp`, `service_name`)
PARTITION BY RANGE(`timestamp`)
(PARTITION p20260907 VALUES [('2026-09-07 00:00:00'), ('2026-09-08 00:00:00')),
PARTITION p20260908 VALUES [('2026-09-08 00:00:00'), ('2026-09-09 00:00:00')),
PARTITION p20260909 VALUES [('2026-09-09 00:00:00'), ('2026-09-10 00:00:00')),
PARTITION p20260910 VALUES [('2026-09-10 00:00:00'), ('2026-09-11 00:00:00')),
PARTITION p20260911 VALUES [('2026-09-11 00:00:00'), ('2026-09-12 00:00:00')),
PARTITION p20260912 VALUES [('2026-09-12 00:00:00'), ('2026-09-13 00:00:00')),
PARTITION p20260913 VALUES [('2026-09-13 00:00:00'), ('2026-09-14 00:00:00')),
PARTITION p20260914 VALUES [('2026-09-14 00:00:00'), ('2026-09-15 00:00:00')),
PARTITION p20260915 VALUES [('2026-09-15 00:00:00'), ('2026-09-16 00:00:00')),
PARTITION p20260916 VALUES [('2026-09-16 00:00:00'), ('2026-09-17 00:00:00')),
PARTITION p20260917 VALUES [('2026-09-17 00:00:00'), ('2026-09-18 00:00:00')),
PARTITION p20260918 VALUES [('2026-09-18 00:00:00'), ('2026-09-19 00:00:00')))
DISTRIBUTED BY HASH(`service_name`) BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "DAY",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.start" = "-10",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "10",
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"enable_single_replica_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728"
);
""", title='Aggregate table for ERROR logs')


0

The `otel_logs_for_error` table adds a `count_value` column at the end. It represents the number of times the same `service_name` occurred on the same day, which accelerates statistical queries.

With the `otel_logs_for_error` table, you can determine how many errors a specific `service_name` generated on a specific day directly from `count_value`. This avoids manually performing the aggregation and significantly improves query performance.

## 2. Unique model for alert deduplication

Consider another log scenario. If ERROR logs are used to trigger alerts for system monitoring, a large number of identical logs can cause an alert storm and affect the delivery of useful alerts. In this scenario, you can use the unique model to prevent large numbers of identical error logs from being written to the table and generating excessive identical alerts.

In [3]:
lab.execute(r"""
CREATE TABLE `otel_logs_for_unique_log` (
  `timestamp` date NULL,
  `alarm_hash` varchar(200) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200) NULL,
  `trace_id` varchar(200) NULL,
  `span_id` text NULL,
  `severity_number` int NULL,
  `severity_text` text NULL,
  `body` text NULL,
  `resource_attributes` variant NULL,
  `log_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL
) ENGINE=OLAP
UNIQUE KEY(`timestamp`, `alarm_hash`)
PARTITION BY RANGE(`timestamp`)
(PARTITION p20260907 VALUES [('2026-09-07 00:00:00'), ('2026-09-08 00:00:00')),
PARTITION p20260908 VALUES [('2026-09-08 00:00:00'), ('2026-09-09 00:00:00')),
PARTITION p20260909 VALUES [('2026-09-09 00:00:00'), ('2026-09-10 00:00:00')),
PARTITION p20260910 VALUES [('2026-09-10 00:00:00'), ('2026-09-11 00:00:00')),
PARTITION p20260911 VALUES [('2026-09-11 00:00:00'), ('2026-09-12 00:00:00')),
PARTITION p20260912 VALUES [('2026-09-12 00:00:00'), ('2026-09-13 00:00:00')),
PARTITION p20260913 VALUES [('2026-09-13 00:00:00'), ('2026-09-14 00:00:00')),
PARTITION p20260914 VALUES [('2026-09-14 00:00:00'), ('2026-09-15 00:00:00')),
PARTITION p20260915 VALUES [('2026-09-15 00:00:00'), ('2026-09-16 00:00:00')),
PARTITION p20260916 VALUES [('2026-09-16 00:00:00'), ('2026-09-17 00:00:00')),
PARTITION p20260917 VALUES [('2026-09-17 00:00:00'), ('2026-09-18 00:00:00')),
PARTITION p20260918 VALUES [('2026-09-18 00:00:00'), ('2026-09-19 00:00:00')))
DISTRIBUTED BY HASH(`alarm_hash`) BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "DAY",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.start" = "-10",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "10",
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"disable_auto_compaction" = "false",
"enable_single_replica_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728"
);
""", title='Unique table for alert deduplication')


0

The `otel_logs_for_unique_log` table contains a column named `alarm_hash`. This field represents the unique value used by the alerting engine, such as a combination of the cluster name, node, and error message. The same `alarm_hash` appears only once per day, which prevents an alert storm.

In summary, unless there is a specific requirement, keep the DUPLICATE model because it can accommodate the vast majority of observability scenarios.